# Lecture Notebook: Correlations, and Reading a Regression Table

**CP201A Planning Methods Gateway, Fall 2026 · Monday, November 23**

This notebook adds one more test to the three from November 18: a **correlation**, for two variables that are both numbers (or 1 to 5 scores). It also has an **optional** test for comparing averages across three or more groups (a one-way ANOVA), and a short section on **reading a regression table**, so the tables you'll see on November 30 and in published research make sense.

### How this notebook fits with the others

| Step | Where | What you get |
|---|---|---|
| 1. Clean the data and make your variables | **Lab 12** notebook | `survey_cleaned_2026.csv` (P/NP #11, Part 1) |
| 2. t-test, Z-test, chi-square | **November 18** notebook, Section 8 | Tests, charts, and draft sentences |
| 3. Correlation (and, optionally, ANOVA) | **This notebook**, Section 8 | The same, for two number variables |
| 4. Multiple regression (optional, advanced) | **Optional Regression Module** notebook, in this folder | A regression table for your final memo |

Every test function from November 18 is also in this notebook, so you can run all of your tests here if you like.

**If you missed class:** Sections 3 to 6 are the correlation examples from lecture (each one goes describe, plot, then test), Section 7 is the optional ANOVA, and Section 9 is the regression table. Section 10 is for your descriptive findings and methods section: shares and means with their n, a "who responded" table, and row and column percents. The slides are on bCourses.

**What you submit:** nothing from this notebook. You copy its results into the P/NP #11 form, due **Sunday, November 29, at 11:59 p.m.**

## 1. Setup: load the data

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.formula.api as smf

pd.set_option('display.float_format', '{:.2f}'.format)

Run the next cell. It looks for **your** cleaned file from Lab 12 (`survey_cleaned_2026.csv`), first in this folder and then in your Lab 12 folder. If it finds it, you're set. If not, it cleans the raw class file the same way Lab 12 does, using the same variable names, so the examples still run.

> **Use your own cleaned file for your own tests.** If the message below says it cleaned the raw file, you're looking at the *demo's* cleaning choices, not yours. That's fine for following along. For P/NP #11 and your final memo, finish Lab 12, save `survey_cleaned_2026.csv`, and rerun this cell until it says **"Loaded YOUR cleaned Lab 12 file."**

The last line makes `city4`, with Richmond and El Cerrito combined, the same way we did on November 18.

In [ ]:
def clean_raw_survey(raw):
    """The Lab 12 cleaning steps, condensed. See Lab 12 for what each step does and why."""
    d = raw[raw['Unnamed: 0'] != 'Example'].drop(columns=['Unnamed: 0'])
    d = d[d['City'].notna()].reset_index(drop=True)
    d = d.map(lambda v: v.strip() if isinstance(v, str) else v)
    d['survey_id'] = d['Surveyor'] + '_' + d['Survey Number'].astype(str)
    d = d.rename(columns={
        'Interviewer or self-administered': 'mode', 'City': 'city', 'Survey Language': 'language',
        'What best describes your role here?': 'role', 'What type of place is this?': 'place_type',
        'About how many years has this business or organization been at this location?': 'years_at_location',
        'About how many years have you worked or volunteered here?': 'years_worked',
        'Including you, about how many people work here?': 'num_workers',
        'Does this business or organization own or rent its space?': 'space_tenure',
        'Do you live in this neighborhood?': 'lives_here', 'How many years have you lived here?': 'years_lived',
        'Do you own or rent your home?': 'home_tenure',
        'On a typical workday, about how many minutes does it take you to get here?': 'commute_min',
        'How do you usually get here?': 'travel_mode',
        'This neighborhood has changed a lot in the past five years.': 'changed_a_lot',
        'Changes in this neighborhood in the past five years have been good for long-time residents.': 'good_for_longtime',
        'Changes in this neighborhood in the past five years have been good for businesses and organizations like this one.': 'good_for_business',
        'Long-time residents can afford to stay in this neighborhood.': 'afford_to_stay',
        'I support building more housing in this neighborhood.': 'support_housing',
        'I feel connected to the people and community in this neighborhood.': 'connected',
        'I feel safe walking alone at night in this neighborhood.': 'safe_at_night',
        'The number of customers or clients who come here': 'p12_customers',
        "This place's sales, revenue, or funding": 'p12_sales',
        "Rent for this business or organization's space": 'p12_space_rent',
        'Other costs of running this place (supplies, wages, utilities)': 'p12_other_costs',
        'Your own rent or mortgage payment': 'p12_own_housing',
        'Prices of everyday goods and services in this neighborhood': 'p12_prices'})

    def looks_like_date(x):
        return isinstance(x, str) and (re.match(r'^\d{4}-\d{2}-\d{2}', x) is not None
                                       or re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}', x) is not None)
    def to_years(x):
        if pd.isna(x): return np.nan
        if isinstance(x, (int, float)): return float(x)
        s = str(x).lower()
        if looks_like_date(s) or 'know' in s or s in ['n/a', 'na', 'skipped', 'did not answer']: return np.nan
        m = re.match(r'^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*$', s)
        if m: return (float(m.group(1)) + float(m.group(2))) / 2
        total, found = 0, False
        for number, unit in re.findall(r'(\d+(?:\.\d+)?)\s*(year|yr|month|mo|week|day)?', s):
            found = True
            v = float(number)
            total += v / 12 if unit.startswith('mo') else v / 52 if unit == 'week' else v / 365 if unit == 'day' else v
        return total if found else np.nan
    def to_minutes(x):
        if pd.isna(x): return np.nan
        if isinstance(x, (int, float)): return float(x)
        s = str(x).lower()
        if looks_like_date(s) or 'know' in s or 'skip' in s: return np.nan
        m = re.match(r'^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)', s)
        if m: v = (float(m.group(1)) + float(m.group(2))) / 2
        else:
            nums = re.findall(r'\d+(?:\.\d+)?', s)
            if not nums: return np.nan
            v = float(nums[0])
        return v * 60 if ('hour' in s or 'hr' in s) else v

    for col in ['years_worked', 'years_at_location', 'years_lived', 'num_workers']:
        d[col + '_num'] = d[col].apply(to_years)
    d['commute_num'] = d['commute_min'].apply(to_minutes)
    agree_dummy = {'Strongly agree': 1, 'Agree': 1, 'Neutral': 0, 'Disagree': 0, 'Strongly disagree': 0}
    agree_score = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5}
    for col in ['changed_a_lot', 'good_for_longtime', 'good_for_business', 'afford_to_stay',
                'support_housing', 'connected', 'safe_at_night']:
        d[col + '_agree_dv'] = d[col].map(agree_dummy)
        d[col + '_score'] = d[col].map(agree_score)
    increased_dummy = {'Increased a lot': 1, 'Increased a little': 1, 'Stayed about the same': 0,
                       'Decreased a little': 0, 'Decreased a lot': 0}
    change_score = {'Decreased a lot': 1, 'Decreased a little': 2, 'Stayed about the same': 3,
                    'Increased a little': 4, 'Increased a lot': 5}
    for col in ['p12_customers', 'p12_sales', 'p12_space_rent', 'p12_other_costs', 'p12_own_housing', 'p12_prices']:
        d[col + '_increased_dv'] = d[col].map(increased_dummy)
        d[col + '_score'] = d[col].map(change_score)
    d['lives_here_dv'] = d['lives_here'].map({'Yes': 1, 'No': 0})
    d['rent_home_dv'] = d['home_tenure'].map({'Rent': 1, 'Own': 0})
    d['space_rented_dv'] = d['space_tenure'].map({'Rent': 1, 'Own': 0})
    d['owner_dv'] = np.where(d['role'].isna(), np.nan, (d['role'] == 'Owner').astype(float))
    d['self_admin_dv'] = d['mode'].map({'Self-administered': 1, 'Interviewer': 0})
    d['drives_alone_dv'] = np.where(d['travel_mode'].isna(), np.nan,
                                    d['travel_mode'].str.contains('Drive alone').astype(float))
    d['transit_dv'] = np.where(d['travel_mode'].isna(), np.nan,
                               d['travel_mode'].str.contains('Public transit').astype(float))
    return d

df = None
for path in ['survey_cleaned_2026.csv', '../Week13_Lab_CleaningSurveyData/survey_cleaned_2026.csv']:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded YOUR cleaned Lab 12 file: {path} ({len(df)} surveys).')
        break
if df is None:
    for path in ['CP201A_2026_survey_data.csv', '../Week13_Lab_CleaningSurveyData/CP201A_2026_survey_data.csv',
                 '../Week13_Lecture_StatisticalTests/CP201A_2026_survey_data.csv']:
        if os.path.exists(path):
            df = clean_raw_survey(pd.read_csv(path))
            print(f'No Lab 12 file found, so we cleaned the raw class file the demo way: {len(df)} surveys.')
            print("NOTE: these are the DEMO's cleaning choices, not yours. Fine for following along with the lecture.")
            print('For P/NP #11 and your final memo, finish Lab 12, save survey_cleaned_2026.csv, and rerun this cell.')
            break
if df is None:
    print('Could not find the data. Upload CP201A_2026_survey_data.csv (from bCourses) to this folder and rerun.')
else:
    # Richmond and El Cerrito combined, because El Cerrito has only a few surveys (as on November 18)
    df['city4'] = df['city'].replace({'Richmond': 'Richmond or El Cerrito', 'El Cerrito': 'Richmond or El Cerrito'})

## 2. The test functions (run this cell; you don't need to change it)

The first five functions are the same ones you used on November 18 (`z_test_shares`, `t_test_means`, `chi_square`, `bar_chart_with_moe`, and `who_responded`). Five are new today:

- `correlate(df, Y, X, method)`: the correlation between two number variables, its p-value, and a draft sentence
- `scatter_plot(df, Y, X, ...)`: the chart that goes with a correlation
- `correlation_table(df, [list of columns])`: correlations among several variables at once
- `anova_means(df, Y, X)` (optional): compares averages across three or more groups
- `describe_variable(df, column)`: a share or a mean with its n, for your descriptive findings (Section 10)

One more, `regression_table(model)`, tidies a regression into a table you can read (Section 9).

**Set your confidence level once, before you look at any results.** Change `CONFIDENCE` in the first line if you want 90 or 99 percent.

In [ ]:
CONFIDENCE = 0.95          # choose 0.90, 0.95, or 0.99 BEFORE looking at results
ALPHA = 1 - CONFIDENCE

def ptext(p):
    return 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'

def sig_words(p):
    return 'statistically significant' if p < ALPHA else 'not statistically significant'

def verdict(p):
    return (f'significant at the {CONFIDENCE:.0%} level ({ptext(p)})' if p < ALPHA
            else f'not significant at the {CONFIDENCE:.0%} level ({ptext(p)})')

# ---------- From November 18 (unchanged) ----------

def z_test_shares(data, y, x, labels=None):
    """Z-test of proportions. y: a 1/0 dummy outcome. x: a 1/0 dummy for two groups."""
    d = data[[y, x]].dropna()
    labels = labels or {0: f'{x} = 0', 1: f'{x} = 1'}
    count = d.groupby(x)[y].sum()
    n = d.groupby(x)[y].count()
    share = count / n
    z, p = proportions_ztest(count[::-1], n[::-1])   # group 1 minus group 0, so the sign matches the difference
    for g in n.index:
        print(f'  {labels[g]:>22}: {share[g]:.1%} (n = {n[g]})')
    gap = (share.iloc[1] - share.iloc[0]) * 100
    print(f'  Difference: {gap:+.1f} percentage points.  Z = {z:.2f}, {verdict(p)}')
    g0, g1 = n.index[0], n.index[1]
    print(f'\n  DRAFT SENTENCE (edit the wording): About {share[g1]:.0%} of {labels[g1]} (n = {n[g1]}) [describe Y], '
          f'compared with {share[g0]:.0%} of {labels[g0]} (n = {n[g0]}), a gap of {abs(gap):.0f} percentage points. '
          f'The difference was {sig_words(p)} at the {CONFIDENCE:.0%} level (Z = {z:.2f}, {ptext(p)}).')
    return share, n, z, p

def t_test_means(data, y, x, labels=None):
    """Welch t-test of two means. y: a number or 1 to 5 score. x: a 1/0 dummy for two groups."""
    d = data[[y, x]].dropna()
    labels = labels or {0: f'{x} = 0', 1: f'{x} = 1'}
    g0, g1 = d[d[x] == 0][y], d[d[x] == 1][y]
    t, p = stats.ttest_ind(g1, g0, equal_var=False)
    for g, vals in [(0, g0), (1, g1)]:
        print(f'  {labels[g]:>22}: mean {vals.mean():.2f}, median {vals.median():.2f} (n = {len(vals)})')
    print(f'  Difference in means: {g1.mean() - g0.mean():+.2f}.  t = {t:.2f}, {verdict(p)}')
    print(f'\n  DRAFT SENTENCE (edit the wording): {labels[1]} averaged {g1.mean():.1f} [units of Y] (n = {len(g1)}), '
          f'compared with {g0.mean():.1f} for {labels[0]} (n = {len(g0)}). '
          f'The difference was {sig_words(p)} at the {CONFIDENCE:.0%} level (Welch t = {t:.2f}, {ptext(p)}).')
    return t, p

def chi_square(data, y, x):
    """Chi-square test for a crosstab of two categorical variables."""
    d = data[[y, x]].dropna()
    table = pd.crosstab(d[x], d[y])
    chi2, p, dof, expected = chi2_contingency(table, correction=False)
    print((pd.crosstab(d[x], d[y], normalize='index') * 100).round(1).to_string())
    print(f'  Chi-square = {chi2:.2f}, df = {dof}, {verdict(p)}')
    small = (expected < 5).sum()
    if small > 0:
        print(f'  Caution: {small} cell(s) have an expected count under 5. Consider combining categories.')
    return chi2, p, dof

def bar_chart_with_moe(data, y, x, labels, title, ylabel):
    """Bar chart of a share (if y is 1/0) or a mean (otherwise) for each group, with margins of error at your CONFIDENCE level."""
    d = data[[y, x]].dropna()
    grp = d.groupby(x)[y]
    is_share = set(d[y].unique()) <= {0, 1}
    z_crit = stats.norm.ppf(1 - ALPHA / 2)   # 1.645 at 90%, 1.96 at 95%, 2.576 at 99%
    center = grp.mean() * (100 if is_share else 1)
    if is_share:
        p_ = grp.mean()
        moe = z_crit * np.sqrt(p_ * (1 - p_) / grp.count()) * 100
    else:
        moe = z_crit * grp.std() / np.sqrt(grp.count())
    names = [f'{labels[g]} (n = {grp.count()[g]})' for g in center.index]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(names, center.values, yerr=moe.values, capsize=6, color=['#5B7FA8', '#2A9D8F', '#F2B134', '#E85C34'][:len(names)])
    if is_share:
        ax.set_ylim(0, 100)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    for i, (c_, m_) in enumerate(zip(center.values, moe.values)):
        ax.text(i, c_ + m_ + (2 if is_share else 0.02 * center.max()), f'{c_:.0f}%' if is_share else f'{c_:.1f}', ha='center')
    plt.tight_layout()
    plt.show()
    print(f'For your Data Note: error bars are {CONFIDENCE:.0%} margins of error, which assume a random sample.')

def who_responded(data, columns):
    """Counts and shares for a few columns: a "who responded" table for your methods section."""
    for col in columns:
        t = data[col].value_counts(dropna=False).rename('count').to_frame()
        t['percent'] = (t['count'] / t['count'].sum() * 100).round(1)
        print(f'\n{col}')
        print(t.to_string())


# ---------- New for November 23 ----------

def strength(r):
    """Rough words for the size of a correlation. Rules of thumb vary by field; say which one you use."""
    a = abs(r)
    return 'very weak' if a < 0.1 else 'weak' if a < 0.3 else 'moderate' if a < 0.5 else 'strong'

def correlate(data, y, x, method='pearson', labels=None):
    """Correlation between two number (or 1 to 5 score) variables.
    method='pearson' for two numbers; method='spearman' for scores, or numbers with a few very large values."""
    d = data[[y, x]].dropna()
    labels = labels or {x: x, y: y}
    if method == 'pearson':
        r, p = stats.pearsonr(d[x], d[y])
        name = 'Pearson r'
    else:
        r, p = stats.spearmanr(d[x], d[y])
        name = "Spearman's rho"
    direction = 'positive' if r > 0 else 'negative'
    print(f'  n = {len(d)} people answered both questions')
    print(f'  {name} = {r:.2f} ({strength(r)}, {direction}); {verdict(p)}')
    if method == 'pearson':
        print(f'  r squared = {r**2:.2f}: about {r**2:.0%} of the variation in {labels[y]} goes along with {labels[x]}')
    print(f'\n  DRAFT SENTENCE (edit the wording): Among the {len(d)} respondents who answered both questions, '
          f'{labels[x]} and {labels[y]} had a {strength(r)} {direction} correlation ({name} = {r:.2f}). '
          f'The correlation was {sig_words(p)} at the {CONFIDENCE:.0%} level ({ptext(p)}).')
    return r, p, len(d)

def scatter_plot(data, y, x, xlabel, ylabel, title, jitter=False, line=True):
    """Scatterplot of two number variables. jitter=True spreads out dots that sit on top of each other
    (useful for 1 to 5 scores); it moves dots a little for the picture only and never changes the data."""
    d = data[[y, x]].dropna()
    xs, ys = d[x].astype(float), d[y].astype(float)
    if jitter:
        rng = np.random.default_rng(0)
        xs = xs + rng.uniform(-0.2, 0.2, len(xs))
        ys = ys + rng.uniform(-0.2, 0.2, len(ys))
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.scatter(xs, ys, alpha=0.45, color='#1B4A82', edgecolor='white', linewidth=0.5)
    if line:
        slope, intercept = np.polyfit(d[x], d[y], 1)
        grid = np.linspace(d[x].min(), d[x].max(), 50)
        ax.plot(grid, intercept + slope * grid, color='#E85C34', linewidth=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    print(f'For your Data Note: n = {len(d)} respondents who answered both questions.'
          + (' Dots are jittered (moved slightly) so overlapping answers show.' if jitter else ''))

def correlation_table(data, columns, method='spearman'):
    """A table of correlations among several variables, with the smallest n used."""
    d = data[columns]
    table = d.corr(method=method)
    n_min = min(d[[a, b]].dropna().shape[0] for a in columns for b in columns if a != b)
    print(f'{method.title()} correlations (each pair uses everyone who answered both; smallest n = {n_min})')
    return table.round(2)

def anova_means(data, y, x):
    """One-way ANOVA (optional): compares the average of y across three or more groups in x."""
    d = data[[y, x]].dropna()
    summary = d.groupby(x)[y].agg(['count', 'mean', 'median']).rename(columns={'count': 'n'})
    print(summary.round(1).to_string())
    groups = [g[y].values for _, g in d.groupby(x)]
    f, p = stats.f_oneway(*groups)
    print(f'  F = {f:.2f}, {verdict(p)}')
    if (summary['n'] < 10).any():
        print('  Caution: at least one group has fewer than 10 people. Consider combining groups.')
    hi, lo = summary['mean'].idxmax(), summary['mean'].idxmin()
    print(f'\n  DRAFT SENTENCE (edit the wording): Average {y} [describe Y] ranged from {summary.loc[lo, "mean"]:.1f} '
          f'for {lo} (n = {summary.loc[lo, "n"]}) to {summary.loc[hi, "mean"]:.1f} for {hi} (n = {summary.loc[hi, "n"]}). '
          f'The differences across the {len(summary)} groups were {sig_words(p)} at the {CONFIDENCE:.0%} level '
          f'(one-way ANOVA, F = {f:.2f}, {ptext(p)}).')
    return f, p

def describe_variable(data, col):
    """A descriptive finding for one variable: the share (for a 0/1 dummy) or the mean and median
    (for a number or score), with the n who answered and how many were left out."""
    vals = data[col]
    answered, left_out = vals.notna().sum(), vals.isna().sum()
    if set(vals.dropna().unique()) <= {0, 1}:
        print(f'  {col}: {vals.mean():.1%} (n = {answered} who answered; {left_out} left out: blank, or coded as missing in cleaning, like "don\'t know")')
        print(f'\n  DRAFT SENTENCE (edit the wording): About {vals.mean():.0%} of the {answered} respondents who answered '
              f'[describe what a 1 means]. ({left_out} respondents left it blank or gave an answer coded as missing, such as "don\'t know.")')
    else:
        print(f'  {col}: mean {vals.mean():.1f}, median {vals.median():.1f} (n = {answered} who answered; {left_out} left out)')
        print(f'\n  DRAFT SENTENCE (edit the wording): Among the {answered} respondents who answered, the average was '
              f'{vals.mean():.1f} [units] and the median was {vals.median():.1f}.')

def regression_table(model):
    """A tidy regression table: one row per variable, plus n and R squared."""
    level = int(round(CONFIDENCE * 100))
    ci = model.conf_int(alpha=ALPHA)
    t = pd.DataFrame({'coefficient': model.params, 'std. error': model.bse,
                      f'{level}% CI low': ci[0], f'{level}% CI high': ci[1], 'p-value': model.pvalues})
    t['significant?'] = np.where(t['p-value'] < ALPHA, f'yes, at {level}%', 'no')
    print(f'n = {int(model.nobs)}   R squared = {model.rsquared:.3f}'
          + ('   (robust standard errors)' if model.cov_type != 'nonrobust' else ''))
    return t.round(3)

## 3. What a correlation measures

A **correlation coefficient** (r) is one number, from −1 to +1, that says how closely two number variables move together **in a straight line**.

- **Positive (r above 0):** the two variables tend to move in the same direction. As one increases, the other tends to increase (and as one decreases, the other tends to decrease). For example, more hours of study often go with higher test scores.
- **Negative (r below 0):** the two variables tend to move in opposite directions. As one increases, the other tends to decrease. For example, higher prices often go with lower sales.
- **r near 0:** no straight-line pattern.

The sign tells you the **direction**. The size (ignoring the sign) tells you the **strength**. Rules of thumb for strength vary by field; this notebook uses a common one: under 0.1 very weak, 0.1 to 0.3 weak, 0.3 to 0.5 moderate, 0.5 and up strong. Whatever rule you use, say so in your Data Note.

The next cell makes made-up data (not our survey) so you can see what different values of r look like.

In [ ]:
# Made-up data, to see what r looks like. Not our survey.
rng = np.random.default_rng(23)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, target in zip(axes, [-0.8, -0.3, 0.0, 0.3, 0.8]):
    x = rng.normal(size=150)
    y = target * x + np.sqrt(1 - target**2) * rng.normal(size=150)
    ax.scatter(x, y, alpha=0.5, s=12, color='#1B4A82')
    ax.set_title(f'r = {np.corrcoef(x, y)[0, 1]:.2f}')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

Two things to notice. A correlation of 0.3 is easy to miss by eye, and yet with 250 people it is very unlikely to be chance. And a correlation only measures a **straight-line** pattern: a U-shaped pattern can have r near 0.

**Every example below follows the same order: describe, then plot, then test.** Look at the numbers and the picture before you run `correlate`.

## 4. Correlation with numbers

### 4.1 Two numbers: years lived here and commute time

**Question:** among respondents who live in the neighborhood where they work, do people who have lived there longer have shorter commutes? (If longtime residents tend to work close to home, we'd expect a negative correlation.)
- X: years lived in the neighborhood (the follow-up to Q7, "How many years have you lived here?"; only residents answered), a number
- Y: "On a typical workday, about how many minutes does it take you to get here?" (Q9), a number

Because only residents answered the years-lived question, `correlate` automatically uses only them (everyone else is missing on X).

**Step 1: describe.** Check the minimum and maximum of each variable.

In [ ]:
df[['years_lived_num', 'commute_num']].dropna().describe()

**Step 2: plot.**

In [ ]:
scatter_plot(df, 'commute_num', 'years_lived_num',
             xlabel='Years lived in the neighborhood (Q7)', ylabel='Commute on a typical workday, minutes (Q9)',
             title='Commute time, by years lived in the neighborhood (residents)')

**Step 3: test.**

In [ ]:
correlate(df, 'commute_num', 'years_lived_num', method='pearson',
          labels={'years_lived_num': 'years lived in the neighborhood',
                  'commute_num': 'commute time'});

A weak result that isn't significant is still a finding. Write it the way you'd write any non-significant result: give r and the n, and say the data can't detect a relationship (not that there isn't one).

### 4.2 Choosing between Pearson and Spearman

**Pearson's r** uses the actual values, so a few very large values can pull it up or down. **Spearman's rho** ranks everyone from lowest to highest first and correlates the ranks, so one extreme value counts as "the highest" and no more.

Common practice:
- Use **Pearson** when both variables are real numbers on an even scale (minutes, years, dollars), the scatterplot looks roughly like a straight line, and there are no extreme values.
- Use **Spearman** when either variable is a **1 to 5 score** (the distance from "agree" to "strongly agree" isn't a number of anything), when a number has a **few very large values** (a long right tail, like years at a location or number of workers), or when the pattern goes steadily up or down without being a straight line. Many survey researchers report Spearman for agree-disagree items like ours.

If Pearson and Spearman tell the same story, good. If they don't, a few people are driving the Pearson result, and you should say so. Either way, name the one you used in your Data Note.

In [ ]:
correlate(df, 'commute_num', 'years_lived_num', method='spearman',
          labels={'years_lived_num': 'years lived in the neighborhood',
                  'commute_num': 'commute time'});

### 4.3 A number and a score: years worked and feeling connected

**Hypothesis:** people who have worked at a place longer feel more connected to the community around it (a positive correlation).
- X: "About how many years have you worked or volunteered here?" (Q6), a number
- Y: "I feel connected to the people and community in this neighborhood." (2f), a 1 to 5 score

Y is a score, so we use Spearman. In the plot, the dots sit in five rows, one for each answer: that's what a score looks like. `jitter=True` spreads them out a little so you can see how many people are in each row.

In [ ]:
df[['years_worked_num', 'connected_score']].dropna().describe()

In [ ]:
scatter_plot(df, 'connected_score', 'years_worked_num',
             xlabel='Years worked or volunteered here (Q6)', ylabel='Feel connected (2f), 1 to 5',
             title='Feeling connected, by years worked here', jitter=True)

In [ ]:
correlate(df, 'connected_score', 'years_worked_num', method='spearman',
          labels={'years_worked_num': 'years worked here',
                  'connected_score': 'feeling connected to the community'});

## 5. Correlation between two 1 to 5 scores

**Hypothesis:** people who agree that changes have been good for businesses like theirs are also more likely to agree that changes have been good for long-time residents (a positive correlation).
- X: "Changes in this neighborhood in the past five years have been good for businesses and organizations like this one." (2c), `good_for_business_score`
- Y: "Changes in this neighborhood in the past five years have been good for long-time residents." (2b), `good_for_longtime_score`

Both run from 1 (strongly disagree) to 5 (strongly agree), so we use Spearman. Scores also pile up on the same few values (there are only 25 possible pairs), so the scatterplot uses `jitter=True`. **Where it goes:** add `jitter=True` as the last item inside `scatter_plot( ... )`, after the title, as in the next cell. The jitter is for the picture only; the correlation uses the real answers.

In [ ]:
scatter_plot(df, 'good_for_longtime_score', 'good_for_business_score',
             xlabel='Changes good for businesses like this one (2c), 1 to 5',
             ylabel='Changes good for long-time residents (2b), 1 to 5',
             title='Two views of neighborhood change', jitter=True)

In [ ]:
correlate(df, 'good_for_longtime_score', 'good_for_business_score', method='spearman',
          labels={'good_for_business_score': 'agreeing that changes have been good for businesses like theirs',
                  'good_for_longtime_score': 'agreeing that changes have been good for long-time residents'});

### 5.1 Many correlations at once: explore, then decide

`correlation_table` shows the correlation for every pair in a list: here, all seven statements in section 2 of the survey (2a to 2g).

**Exploring your data is good practice.** Looking at tables like this one, along with your cleaning checks, is how you spot outliers, odd patterns, and relationships you didn't expect, and it can sharpen your question or lead you to a better one.

**The line to watch is between exploring and fishing.** With 21 pairs, one or two can look significant by chance alone (the coin-flip cell from November 18). If you pick the one pair that came out significant and write it up as if you had predicted it, you've turned noise into a finding (sometimes called p-hacking). The honest version: if a hypothesis came from exploring, say so in your memo, explain why it matters for planning, and report it alongside the hypotheses you started with, significant or not.

In [ ]:
grid = ['changed_a_lot_score', 'good_for_longtime_score', 'good_for_business_score', 'afford_to_stay_score',
        'support_housing_score', 'connected_score', 'safe_at_night_score']   # statements 2a to 2g
correlation_table(df, grid, method='spearman')

## 6. Three cautions before you write it up

**1. Significant is not the same as strong.** With 200 to 250 people, a correlation as small as 0.12 to 0.14 can clear the 95 percent bar. Report r (the size) and the p-value (whether the data can detect it), and describe the size in words.

**2. Correlation is not causation.** A correlation can come from X affecting Y, Y affecting X, or a third thing affecting both. In our survey, **city** and **survey mode** are third things to check, because answers differ by both. The next cell checks whether the Section 5 correlation holds within each city.

**3. Look at the picture.** One unusual respondent, or a curved pattern, can make r misleading. Always make the scatterplot.

In [ ]:
# Does the Section 5 correlation (2c and 2b) hold within each city? (Spearman; small groups give noisy answers)
for c, g in df.groupby('city4'):
    d = g[['good_for_longtime_score', 'good_for_business_score']].dropna()
    rho, p = stats.spearmanr(d['good_for_business_score'], d['good_for_longtime_score'])
    print(f'{c:>24}: rho = {rho:.2f}, n = {len(d)}, {verdict(p)}')

## 7. Optional: comparing averages across three or more groups (one-way ANOVA)

On November 18 we had a test for a number outcome with **two** groups (the t-test) and a test for a **category** outcome with three or more groups (chi-square). That left a gap: a **number** (or score) outcome across **three or more** groups, like commute minutes by city. A one-way **ANOVA** ("analysis of variance") fills it. It asks one question: are the group averages more different from each other than we'd expect by chance?

- It gives an **F statistic** and a p-value. A large F (and small p) means at least one group's average differs from the others. It doesn't say which.
- Standard ANOVA assumes the groups are about equally spread out. If they aren't, check the group standard deviations and say so.
- **No-new-code alternative:** collapse to two groups (for example, Oakland vs. everywhere else) and use `t_test_means`.

**Hypothesis:** average commute times (Q9, minutes on a typical workday) differ by city.

In [ ]:
anova_means(df, 'commute_num', 'city4');

In [ ]:
# Are the groups about equally spread out? Compare the standard deviations.
df.groupby('city4')['commute_num'].std().round(1)

## 8. Your turn: test your own hypotheses, step by step

These are the same steps as Section 8 of the November 18 notebook, with two more rows in the Step 4 table.

**Before you start:** check the message in Section 1. It should say **"Loaded YOUR cleaned Lab 12 file."**

**Step 1. Write your hypothesis** as a `#` comment in the test cell, with the direction you expect.

**Step 2. Find your variable names.** Run the Step 2 cell.

**Step 3. Check your variables** with `.describe()` (numbers and scores) or `value_counts(dropna=False)` (dummies and categories). For a correlation, look at the minimum and maximum: a few very large values are a reason to use Spearman.

**Step 4. Pick the test:**

| Your outcome (Y) is... | Your explanatory variable (X) is... | Use |
|---|---|---|
| a 0/1 dummy | a 0/1 dummy | `z_test_shares(df, Y, X, LABELS)` |
| a number or a 1 to 5 score | a 0/1 dummy | `t_test_means(df, Y, X, LABELS)` |
| a dummy or a category | a category with more than two groups | `chi_square(df, Y, X)` |
| a number or a 1 to 5 score | a category with more than two groups | `anova_means(df, Y, X)` (optional) |
| a number | a number | `correlate(df, Y, X, method='pearson', labels=LABELS)` |
| a 1 to 5 score, or a number with a few very large values | a number or a score | `correlate(df, Y, X, method='spearman', labels=LABELS)` |

**Step 5. Fill in the test cell:** replace the names after `Y =` and `X =` with yours (keep the quote marks). For a correlation, `LABELS` gives a short description of each variable, used in the draft sentence.

**Step 6. Make your chart.** For a correlation, use `scatter_plot` (add `jitter=True` for scores). For the other tests, use `bar_chart_with_moe`. Right-click the chart to save it.

**Step 7. Fill in the P/NP #11 form:** your hypothesis, how you coded Y and X, the test, the n, the result (r or rho, Z, t, chi-square, or F), the p-value, whether it's significant at your level, the chart, and the draft sentence in your own words.

**Pick your hypotheses before you test, and report what you find, significant or not.**

In [ ]:
# Step 2: list the variables you can test
[c for c in df.columns if c.endswith('_dv') or c.endswith('_score') or c.endswith('_num')]

In [ ]:
# Step 3: check your variables. Replace the names with yours.
df[['years_worked_num', 'commute_num']].describe()

In [ ]:
# TEST (correlation)
# Step 1, my hypothesis: [write it here, with a direction]
# Step 5: change the next three lines to your variables
Y = 'years_worked_num'                   # your outcome
X = 'commute_num'                        # your explanatory variable
LABELS = {X: 'commute time', Y: 'years worked here'}   # short descriptions for the draft sentence

correlate(df, Y, X, method='spearman', labels=LABELS);

In [ ]:
# Step 6: chart. Change the labels and title. Add jitter=True if either variable is a 1 to 5 score.
scatter_plot(df, Y, X, xlabel='Commute (minutes)', ylabel='Years worked here',
             title='Years worked, by commute time')

In [ ]:
# A SECOND TEST: copy any test line from the Step 4 table here and fill in your variables.
# Step 1, my hypothesis: [write it here, with a direction]

### If you get an error

| What you see | What it usually means | What to do |
|---|---|---|
| `NameError: name 'df' is not defined` (or a function is not defined) | Sections 1 and 2 haven't run since you opened the notebook | Run Sections 1 and 2 again, top to bottom |
| `KeyError: 'some_name'` | A variable name is misspelled, or isn't in your file | Copy the exact name from the Step 2 list; if it's missing, make it in Lab 12 and save again |
| "Could not find the data" | The data file isn't in this folder or your Lab 12 folder | Upload `CP201A_2026_survey_data.csv` (bCourses), or save `survey_cleaned_2026.csv` in Lab 12 |
| A Z-test or t-test error, or a strange result | Your X has more than two values (for example 0, 1, and 2) | Check X with `value_counts(dropna=False)`; for three or more groups, use `chi_square` or `anova_means` |
| A correlation with a much smaller n than you expected | Many people skipped one of the two questions | Report the n you got, and say what was left out |

Still stuck? Copy the error message and the cell into an email to the teaching team, or bring it to office hours. Under the course AI policy you may use an AI tool for coding help; say so if you do.

## 9. Reading a regression table

You don't need to run a regression for P/NP #11. But regression tables are everywhere in planning research, and next Monday we'll read published ones. This section shows that a regression is a close cousin of the tests you already know, using our own data.

A **linear regression** draws the straight line that best fits the data and reports its pieces:

- **Intercept:** the predicted Y when every X is 0
- **Coefficient** on an X: how much the predicted Y changes when that X goes up by 1, *holding the other X's in the model constant*
- **Standard error, confidence interval, p-value:** how precisely the data pin down each coefficient (the same logic as every test so far)
- **n** and **R squared:** how many people the model used, and the share of the variation in Y the model accounts for

We use **robust standard errors** (`cov_type='HC1'`), which don't assume equal spreads, for the same reason we use Welch's t-test.

### 9.1 One dummy X: a regression is a t-test in disguise

Commute minutes (Q9), residents vs. people who live elsewhere (Q7) (the November 18 t-test). In the regression, the **intercept** is the average for people who live elsewhere (`lives_here_dv` = 0), and the **coefficient** on `lives_here_dv` is the difference between the two group averages.

In [ ]:
m1 = smf.ols('commute_num ~ lives_here_dv', data=df).fit(cov_type='HC1')
regression_table(m1)

In [ ]:
# Compare with the November 18 t-test: same difference in means, nearly the same p-value
t_test_means(df, 'commute_num', 'lives_here_dv', labels={0: 'Lives elsewhere', 1: 'Lives here'});

### 9.2 A category X: ANOVA is a regression on group dummies

`C(city4)` tells the regression to make a dummy for each city but one. The city left out is the **reference group** (here, Berkeley, first alphabetically), and each coefficient is that city's average **minus** Berkeley's. The overall F test at the bottom of a full regression output is the same F as the Section 7 ANOVA.

In [ ]:
# See what C(city4) does: one column becomes a 0/1 column for each city but one (Berkeley, first alphabetically).
# You never need to make these yourself for a regression; this just shows what happens inside.
pd.concat([df['city4'], pd.get_dummies(df['city4'], drop_first=True, dtype=int)], axis=1).groupby('city4').head(2).sort_values('city4')

In [ ]:
m2 = smf.ols('commute_num ~ C(city4)', data=df).fit()
print(f'Regression F = {m2.fvalue:.2f}, {verdict(m2.f_pvalue)}   (compare with the Section 7 ANOVA)')
regression_table(m2)

### 9.3 Two X's at once: "holding constant"

Residents' shorter commutes might partly be a city difference (people in some cities may live closer to work). To check, put both in the model. The coefficient on `lives_here_dv` now compares residents and non-residents **in the same city**.

In [ ]:
m3 = smf.ols('commute_num ~ lives_here_dv + C(city4)', data=df).fit(cov_type='HC1')
regression_table(m3)

**How to read it:** compare the `lives_here_dv` coefficient here with the one in 9.1. If it barely changes, city wasn't behind the residents' gap. If it shrinks a lot, some of the gap was a city difference. Either way, a regression on survey data shows an **association** after accounting for the variables you included, not a cause: anything left out of the model can still matter.

**To run your own,** use the **Optional Regression Module** notebook in this folder. It walks you through a multiple regression step by step, for the advanced option in Part III of the final memo.

## 10. For your descriptive findings and methods section

Part III of the final memo needs at least two **descriptive findings** (shares or means, with the n and what was left out) and a **who responded** table in the methods. This section gives you both, plus a guide to row and column percents.

### 10.1 One variable: a share or a mean, with its n

`describe_variable` works on any dummy (it gives the share) or number or score (it gives the mean and median), and tells you how many people answered and how many were left out.

In [ ]:
describe_variable(df, 'p12_own_housing_increased_dv')   # a dummy: the share who said their rent or mortgage went up
print()
describe_variable(df, 'commute_num')                     # a number: average and median commute

In [ ]:
# EXERCISE #1: Use describe_variable on the variables for YOUR two descriptive findings.
# Then write one sentence for each as a # comment, with the n and what was left out.

### 10.2 Who responded (for your methods section)

Change the list to the columns you want to describe. Report the counts and percents in a small table, and say the survey was not a random sample.

In [ ]:
who_responded(df, ['city', 'role', 'mode', 'language'])

### 10.3 Two variables: row percents or column percents?

The same crosstab counts give two different sets of percents, and they answer different questions.

- **Row percents** (`normalize='index'`): each row adds to 100. *Of the people who live in the neighborhood, what share rent?*
- **Column percents** (`normalize='columns'`): each column adds to 100. *Of the renters, what share live in the neighborhood?*

Both are correct; your question decides which one you want. A good habit: put your **X** in the rows and use row percents, so you're comparing the groups of X (that's how the test functions work). Whichever you use, say what the percentages are **out of** in your Data Note.

Start with the counts and their totals (`margins=True`), so you can see each denominator. This example keeps only people who said they own or rent.

In [ ]:
d10 = df[df['home_tenure'].isin(['Own', 'Rent'])]

# Counts, with a total for each row and column
pd.crosstab(d10['lives_here'], d10['home_tenure'], margins=True, margins_name='Total')

In [ ]:
# Row percents: each row adds to 100 (of residents, what share rent?)
(pd.crosstab(d10['lives_here'], d10['home_tenure'], normalize='index') * 100).round(1)

In [ ]:
# Column percents: each column adds to 100 (of renters, what share live here?)
(pd.crosstab(d10['lives_here'], d10['home_tenure'], normalize='columns') * 100).round(1)

In [ ]:
# EXERCISE #2: Make a crosstab for one of YOUR descriptive findings. Put your X in the rows.
# Print the counts with margins=True, then the row percents. In a # comment, write one sentence
# that says what the percentages are out of.

### 10.4 Testing a bigger crosstab: both variables with several categories

Chi-square also works when **both** variables have several categories. Here: the person's role (Q1) by city. It's the same test with a bigger table. Read the row percents to see which cities differ, and watch for the caution about small expected counts, which is more likely with more cells. We combine "Volunteer" and "Other" into one group, and count the one person who checked both employee and manager as a manager.

(Logistic regression is a different tool: it's for a yes-or-no **outcome** when you want several explanatory variables at once. See Section 7.1 of the optional regression module.)

In [ ]:
df['role4'] = df['role'].replace({'Employee, Manager/supervisor': 'Manager/supervisor',
                                  'Volunteer': 'Volunteer or other', 'Other': 'Volunteer or other'})
chi_square(df, 'role4', 'city4');

Notice the caution: a couple of cells have expected counts under 5 (there are few volunteers in some cities). Before you trust this result, combine further (for example, owners and managers vs. employees, volunteers, and others) and test again, or report it with that caveat.